In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
cuadro = [
    cv2.cvtColor(cv2.imread(f'img/cuadro_{i}.jpg'), cv2.COLOR_BGR2RGB)
    for i in range(3)
]

udesa = [
    cv2.cvtColor(cv2.imread(f'img/udesa_{i}.jpg'), cv2.COLOR_BGR2RGB)
    for i in range(3)
]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, img in enumerate(cuadro):
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'cuadro_{i}')
    axes[0, i].axis('off')

for i, img in enumerate(udesa):
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'udesa_{i}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 3.1 Detección y descripción de características visuales (SIFT)

In [ ]:
sift = cv2.SIFT_create()

def detect_features(images):
    """Detecta keypoints y descriptores SIFT para una lista de imágenes."""
    kps_list, descs_list = [], []
    for img in images:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        kps, descs = sift.detectAndCompute(gray, None)
        kps_list.append(kps)
        descs_list.append(descs)
    return kps_list, descs_list

kps_udesa, descs_udesa = detect_features(udesa)
kps_cuadro, descs_cuadro = detect_features(cuadro)

for i in range(3):
    print(f"udesa_{i}: {len(kps_udesa[i])} keypoints")
for i in range(3):
    print(f"cuadro_{i}: {len(kps_cuadro[i])} keypoints")

In [ ]:
def draw_kps(img, kps, min_size=8, color=(255, 220, 0), thickness=2):
    """Dibuja keypoints con tamaño mínimo fijo para mayor visibilidad."""
    out = img.copy()
    for kp in kps:
        x, y = int(kp.pt[0]), int(kp.pt[1])
        r = max(int(kp.size / 2), min_size)
        cv2.circle(out, (x, y), r, color, thickness, cv2.LINE_AA)
    return out

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i in range(3):
    axes[0, i].imshow(draw_kps(udesa[i], kps_udesa[i]))
    axes[0, i].set_title(f'udesa_{i} — {len(kps_udesa[i])} kps')
    axes[0, i].axis('off')

for i in range(3):
    axes[1, i].imshow(draw_kps(cuadro[i], kps_cuadro[i]))
    axes[1, i].set_title(f'cuadro_{i} — {len(kps_cuadro[i])} kps')
    axes[1, i].axis('off')

plt.suptitle('3.1 — Keypoints SIFT detectados', fontsize=14)
plt.tight_layout()
plt.show()

## 3.2 Supresión de No Máxima Adaptativa (ANMS)

In [11]:
def anms(keypoints, N):
    """
    Adaptive Non-Maximal Suppression (Algoritmo 1 del enunciado).
    Retorna los N keypoints más distribuidos espacialmente,
    priorizando los de mayor respuesta con mayor radio de supresión.
    """
    if len(keypoints) <= N:
        return keypoints

    # Extraer posiciones y respuestas
    pts = np.array([kp.pt for kp in keypoints], dtype=np.float32)   # (M, 2)
    responses = np.array([kp.response for kp in keypoints])          # (M,)

    M = len(keypoints)
    radii = np.full(M, np.inf)

    for i in range(M):
        for j in range(M):
            if responses[j] > responses[i]:
                sd = (pts[j, 0] - pts[i, 0])**2 + (pts[j, 1] - pts[i, 1])**2
                if sd < radii[i]:
                    radii[i] = sd

    # Ordenar por radio descendente y quedarse con los N mejores
    top_idx = np.argsort(radii)[::-1][:N]
    return [keypoints[i] for i in top_idx]


N = 500  # máximo de keypoints por imagen

kps_udesa_anms  = [anms(kps, N) for kps in kps_udesa]
kps_cuadro_anms = [anms(kps, N) for kps in kps_cuadro]

# Filtrar descriptores para que coincidan con los kps seleccionados
def filter_descs(kps_orig, kps_filtered, descs):
    """Devuelve los descriptores correspondientes a kps_filtered."""
    pts_orig     = [kp.pt for kp in kps_orig]
    pts_filtered = {kp.pt for kp in kps_filtered}
    idx = [i for i, pt in enumerate(pts_orig) if pt in pts_filtered]
    return descs[idx]

descs_udesa_anms  = [filter_descs(kps_udesa[i],  kps_udesa_anms[i],  descs_udesa[i])  for i in range(3)]
descs_cuadro_anms = [filter_descs(kps_cuadro[i], kps_cuadro_anms[i], descs_cuadro[i]) for i in range(3)]

for i in range(3):
    print(f"udesa_{i}:  {len(kps_udesa[i])} → {len(kps_udesa_anms[i])} kps")
for i in range(3):
    print(f"cuadro_{i}: {len(kps_cuadro[i])} → {len(kps_cuadro_anms[i])} kps")

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\Lucas\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py", line 3460, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\Lucas\AppData\Local\Temp\ipykernel_13004\3625329562.py", line 32, in <module>
    kps_cuadro_anms = [anms(kps, N) for kps in kps_cuadro]
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Lucas\AppData\Local\Temp\ipykernel_13004\3625329562.py", line 32, in <listcomp>
    kps_cuadro_anms = [anms(kps, N) for kps in kps_cuadro]
                       ^^^^^^^^^^^^
  File "C:\Users\Lucas\AppData\Local\Temp\ipykernel_13004\3625329562.py", line -1, in anms
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Lucas\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py", line 2057, in showtraceback
    stb = self.InteractiveTB.st

In [ ]:
# Comparación antes/después de ANMS
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].imshow(draw_kps(udesa[1], kps_udesa[1]))
axes[0, 0].set_title(f'udesa_1 — SIFT ({len(kps_udesa[1])} kps)')
axes[0, 0].axis('off')

axes[0, 1].imshow(draw_kps(udesa[1], kps_udesa_anms[1]))
axes[0, 1].set_title(f'udesa_1 — ANMS ({len(kps_udesa_anms[1])} kps)')
axes[0, 1].axis('off')

axes[1, 0].imshow(draw_kps(cuadro[1], kps_cuadro[1]))
axes[1, 0].set_title(f'cuadro_1 — SIFT ({len(kps_cuadro[1])} kps)')
axes[1, 0].axis('off')

axes[1, 1].imshow(draw_kps(cuadro[1], kps_cuadro_anms[1]))
axes[1, 1].set_title(f'cuadro_1 — ANMS ({len(kps_cuadro_anms[1])} kps)')
axes[1, 1].axis('off')

plt.suptitle('3.2 — SIFT vs ANMS (imagen ancla)', fontsize=14)
plt.tight_layout()
plt.show()

## 3.3 Asociación de características (Matching)

In [ ]:
def lowe_match(desc1, desc2, ratio=0.75):
    """
    Matching con Lowe's ratio test.
    Retorna lista de cv2.DMatch (el mejor match de cada descriptor de desc1).
    """
    bf = cv2.BFMatcher(cv2.NORM_L2)
    knn = bf.knnMatch(desc1, desc2, k=2)
    good = [m for m, n in knn if m.distance < ratio * n.distance]
    return good

# Ancla = imagen _1 en ambos conjuntos
# Pares: (0→1) y (2→1)
matches_udesa_01  = lowe_match(descs_udesa_anms[0],  descs_udesa_anms[1])
matches_udesa_21  = lowe_match(descs_udesa_anms[2],  descs_udesa_anms[1])
matches_cuadro_01 = lowe_match(descs_cuadro_anms[0], descs_cuadro_anms[1])
matches_cuadro_21 = lowe_match(descs_cuadro_anms[2], descs_cuadro_anms[1])

print(f"udesa  0→1: {len(matches_udesa_01)}  matches")
print(f"udesa  2→1: {len(matches_udesa_21)}  matches")
print(f"cuadro 0→1: {len(matches_cuadro_01)} matches")
print(f"cuadro 2→1: {len(matches_cuadro_21)} matches")

In [ ]:
# Visualización de matches (limitados a 50 para claridad)
def show_matches(img1, kps1, img2, kps2, matches, title, max_draw=50):
    drawn = cv2.drawMatches(img1, kps1, img2, kps2,
                            matches[:max_draw], None,
                            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=(16, 5))
    plt.imshow(drawn)
    plt.title(f'{title}  ({len(matches)} matches, mostrando {min(max_draw, len(matches))})')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

show_matches(udesa[0],  kps_udesa_anms[0],  udesa[1],  kps_udesa_anms[1],  matches_udesa_01,  'udesa 0→1')
show_matches(udesa[2],  kps_udesa_anms[2],  udesa[1],  kps_udesa_anms[1],  matches_udesa_21,  'udesa 2→1')
show_matches(cuadro[0], kps_cuadro_anms[0], cuadro[1], kps_cuadro_anms[1], matches_cuadro_01, 'cuadro 0→1')
show_matches(cuadro[2], kps_cuadro_anms[2], cuadro[1], kps_cuadro_anms[1], matches_cuadro_21, 'cuadro 2→1')

## 3.4 Estimación de la homografía "manualmente" (DLT)

In [ ]:
def dlt(pts_src, pts_dst):
    """
    Direct Linear Transformation (sin cv2).
    pts_src, pts_dst: arrays (N, 2), N >= 4.
    Retorna H (3x3) tal que pts_dst ~ H @ pts_src (en coords homogéneas).
    """
    N = len(pts_src)
    A = []
    for (x, y), (xp, yp) in zip(pts_src, pts_dst):
        A.append([-x, -y, -1,  0,  0,  0,  xp*x, xp*y, xp])
        A.append([ 0,  0,  0, -x, -y, -1,  yp*x, yp*y, yp])
    A = np.array(A, dtype=np.float64)
    _, _, Vt = np.linalg.svd(A)
    H = Vt[-1].reshape(3, 3)
    return H / H[2, 2]


# ── Puntos seleccionados manualmente (editor de imágenes) ─────────────────────
# Formato: (x, y) en píxeles — src = imagen lateral, dst = imagen ancla (_1)
# ACTUALIZAR estos valores mirando las imágenes con un visor.

# udesa: 0 → 1
pts_udesa_0_src = np.array([[372, 82], [500, 78], [490, 192], [370, 196]], dtype=np.float64)
pts_udesa_0_dst = np.array([[130, 82], [258, 80], [250, 192], [128, 196]], dtype=np.float64)

# udesa: 2 → 1
pts_udesa_2_src = np.array([[100, 80], [240, 82], [238, 195], [100, 193]], dtype=np.float64)
pts_udesa_2_dst = np.array([[330, 80], [468, 82], [466, 194], [330, 192]], dtype=np.float64)

# cuadro: 0 → 1
pts_cuadro_0_src = np.array([[280, 100], [420, 95], [415, 220], [278, 222]], dtype=np.float64)
pts_cuadro_0_dst = np.array([[ 80,  98], [220, 93], [216, 218], [ 78, 220]], dtype=np.float64)

# cuadro: 2 → 1
pts_cuadro_2_src = np.array([[110,  95], [250,  98], [248, 220], [108, 218]], dtype=np.float64)
pts_cuadro_2_dst = np.array([[310,  96], [450,  98], [448, 218], [308, 216]], dtype=np.float64)

H_dlt_udesa_01  = dlt(pts_udesa_0_src,  pts_udesa_0_dst)
H_dlt_udesa_21  = dlt(pts_udesa_2_src,  pts_udesa_2_dst)
H_dlt_cuadro_01 = dlt(pts_cuadro_0_src, pts_cuadro_0_dst)
H_dlt_cuadro_21 = dlt(pts_cuadro_2_src, pts_cuadro_2_dst)

print("H_dlt udesa 0→1:\n", H_dlt_udesa_01)
print("\nH_dlt udesa 2→1:\n", H_dlt_udesa_21)

In [ ]:
# Verificación visual: warpPerspective con H_dlt
def show_warp(img_src, img_anchor, H, title):
    h, w = img_anchor.shape[:2]
    warped = cv2.warpPerspective(img_src, H, (w, h))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].imshow(img_src);     axes[0].set_title('Fuente');   axes[0].axis('off')
    axes[1].imshow(img_anchor);  axes[1].set_title('Ancla');    axes[1].axis('off')
    axes[2].imshow(warped);      axes[2].set_title('Warped');   axes[2].axis('off')
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

show_warp(udesa[0],  udesa[1],  H_dlt_udesa_01,  '3.4 DLT — udesa 0→1')
show_warp(udesa[2],  udesa[1],  H_dlt_udesa_21,  '3.4 DLT — udesa 2→1')
show_warp(cuadro[0], cuadro[1], H_dlt_cuadro_01, '3.4 DLT — cuadro 0→1')
show_warp(cuadro[2], cuadro[1], H_dlt_cuadro_21, '3.4 DLT — cuadro 2→1')